In [2]:
import os
import re
import pandas as pd
from datetime import datetime

import ee
import geemap



#####################################
#    SET ALL PARAMETERS #############
#####################################

GEE_PROJECT = 'ee-tymc5571-multi-disturbance'

#ee.Authenticate()
ee.Authenticate(
    auth_mode='notebook',  # or 'gcloud' or 'appdefault' in other contexts
    scopes=['https://www.googleapis.com/auth/earthengine',
            'https://www.googleapis.com/auth/devstorage.read_write',
            'https://www.googleapis.com/auth/drive']
    #, quiet=False  # force re-authentication even if cached tokens exist
)
ee.Initialize(project=GEE_PROJECT)

#DRIVE_FOLDER = "GEE_Exports_resilience_wumiweather"
DRIVE_FOLDER = "GEE_Exports_resilience_weltyweather"
fire_date_attr = 'date'; #this is the name of the attribute that holds the fire date


#firesRaw = ee.FeatureCollection("projects/ee-tymc5571-cbi-module/assets/WUMI2024a_main_fires_unified_simple_no_circles")
firesRaw = ee.FeatureCollection("projects/ee-tymc5571-goodfire/assets/welty_wildfire_1984_2020")



print(firesRaw.first().getInfo())

{'type': 'Feature', 'geometry': {'type': 'Polygon', 'coordinates': [[[-114.62472629022763, 47.31962031348382], [-114.6245555178964, 47.3192132642698], [-114.61837177687492, 47.31798826769458], [-114.61056512356676, 47.31944288681828], [-114.60747406862424, 47.32072524270981], [-114.60472711960185, 47.32153947517568], [-114.60000493375567, 47.322705370150544], [-114.59966086614837, 47.32409819446703], [-114.60163707357756, 47.32497321302677], [-114.60798666221781, 47.32497239429426], [-114.61408195096277, 47.32415898737527], [-114.61743144370027, 47.32474108610809], [-114.6212032476732, 47.3254943185274], [-114.62257930933336, 47.32567269347912], [-114.62291914366351, 47.32357529355719], [-114.62421213805781, 47.321828455829575], [-114.62472761178026, 47.32020151890688], [-114.62472629022763, 47.31962031348382]]]}, 'id': '00000000000000000001', 'properties': {'Assigned_F': 'Likely Wildfire', 'Circle_Fla': 0, 'Circleness': 0.612030456369, 'Exclude_Fr': 'No', 'Fire_Attri': '3 (1)', 'Fire_

In [2]:
def add_gridmet_means_by_feature_date(
    fc: ee.FeatureCollection,
    date_col: str,
    vars_: list,
    *,
    out_prefix: str = "gridmet_mean_",
    scale: int = 4000,
    bestEffort: bool = True,
    maxPixels: float = 1e13
) -> ee.FeatureCollection:

    gridmet = ee.ImageCollection("IDAHO_EPSCOR/GRIDMET")

    # Precompute output keys (client-side) to avoid ee.List(None) issues
    out_keys_py = [f"{out_prefix}{v}" for v in vars_]
    out_keys_ee = ee.List(out_keys_py)

    # Precompute the no-data dict (client-side)
    no_data_dict = {k: None for k in out_keys_py}

    vars_list = ee.List(vars_)

    def _map_fn(feat):
        feat = ee.Feature(feat)

        d0 = ee.Date.parse("YYYY-MM-dd", feat.get(date_col))
        d1 = d0.advance(1, "day")

        daily_ic = gridmet.filterDate(d0, d1)
        n_img = daily_ic.size()

        def _has_data():
            img = ee.Image(daily_ic.first()).select(vars_list)

            stats = img.reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=feat.geometry(),
                scale=scale,
                bestEffort=bestEffort,
                maxPixels=maxPixels
            )

            out_vals = vars_list.map(lambda v: stats.get(ee.String(v)))
            out_dict = ee.Dictionary.fromLists(out_keys_ee, out_vals)

            return feat.set(out_dict)

        def _no_data():
            return feat.set(no_data_dict)

        return ee.Feature(ee.Algorithms.If(n_img.gt(0), _has_data(), _no_data()))

    return ee.FeatureCollection(fc.map(_map_fn))


def fc_to_dataframe(fc, n=1000):
    features = fc.limit(n).getInfo()['features']
    rows = [f['properties'] for f in features]
    return pd.DataFrame(rows)

def drop_geometry(fc):
    return fc.map(lambda f: ee.Feature(None).copyProperties(f))


In [3]:
fc_out = add_gridmet_means_by_feature_date(firesRaw,
                                           date_col= fire_date_attr,
                                           vars_ = ["vs", "tmmx", "vpd"],
                                           out_prefix="")

fc_out = drop_geometry(fc_out)

task = ee.batch.Export.table.toDrive(
    collection=fc_out,
    description=f'WUMI2024a_weather',
    folder=DRIVE_FOLDER,
    fileNamePrefix=f'WUMI2024a_weather',
    fileFormat='CSV'
)
task.start()

